# 05 — Central-Chile megadrought: where did the rain go?

[Open in Colab](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/blob/main/notebooks/05_megadrought_central_chile.ipynb)

**What we're investigating.** Central Chile entered an unprecedented
decade-long "megadrought" in 2010 that has persisted into the 2020s.
Beyond the obvious precipitation deficit, did the *source* of the
moisture also change? This notebook compares moisture-source maps for
the historical baseline (1990–2009) against the drought decade
(2010–2019), partitions the inflow into Pacific-oceanic vs
continental-recycled fractions, and asks how the
continental-recycling ratio evolved year by year.

**What you'll get.** Three figures and an inline summary block:

- `fig1_source_maps.png`: a 3-panel map (historical | drought |
  anomaly), with a diverging RdBu colour bar on the anomaly panel.
- `fig2_ocean_land_bar.png`: grouped bar chart, ocean vs continental
  fractions for each period.
- `fig3_annual_stacked_bar.png`: stacked bars (continental + oceanic)
  for every year + the recycling ratio overlaid on a secondary axis.
- Period summary statistics: total moisture deficit, fraction shifts,
  recycling-ratio shift.

**Dataset.** AguaTrack v1, **yearly aggregate** zarr stores
(`{year}_yearly_sum.zarr`). We sum over the receptor box (30-38 deg S,
75-68 deg W) of tags.

**How to cite.** TODO: paper in submission.

## Step 1 — Configuration

Everything you might want to edit lives in this single cell:

- **HuggingFace dataset** — `AguaTrack/AguaTrack-ARCO-SA`. Yearly
  aggregate sub-directory is not yet uploaded.
- **Receptor box** — central Chile (~Santiago south to Puerto Montt).
  Set `LAT_MIN`/`LAT_MAX`/`LON_MIN`/`LON_MAX` to any other South
  American sink to compare drought-decade source shifts there.
- **Period split** — 20 years baseline + 10 years drought, the
  standard split used by the Chilean climate-science community for
  this event.

In [ ]:
HF_REVISION = "main"

LAT_MIN, LAT_MAX = -38.0, -30.0
LON_MIN, LON_MAX = -75.0, -68.0
HISTORICAL_YEARS = list(range(1990, 2010))   # 20 years baseline
DROUGHT_YEARS = list(range(2010, 2020))      # 10 years drought decade
ALL_YEARS = HISTORICAL_YEARS + DROUGHT_YEARS

# Full zarr URLs, one per year. The yearly-aggregate sub-directory is
# not yet on HuggingFace — these URLs will start resolving once it is
# uploaded under the same naming scheme.
AGUATRACK_YEARLY_URLS = [
    f"hf://datasets/AguaTrack/AguaTrack-ARCO-SA/AguaTrack_ARCO_SA_yearly/{yr}_yearly_sum.zarr"
    for yr in ALL_YEARS
]

## Step 2 — Install dependencies (Colab only)

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        'install -q cartopy cmcrameri "xarray>=2026" "zarr>=3" '
        "fsspec huggingface_hub dask",
    )

## Step 3 — Imports and plotting style

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import xarray as xr

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
})

## Step 4 — Find tag cells inside the receptor box

In [ ]:
ds_ref = xr.open_zarr(AGUATRACK_YEARLY_URLS[0],
                      storage_options={"revision": HF_REVISION})
in_box = (
    (ds_ref.tag_lat >= LAT_MIN) & (ds_ref.tag_lat <= LAT_MAX)
    & (ds_ref.tag_lon >= LON_MIN) & (ds_ref.tag_lon <= LON_MAX)
)
box_tag_idx = np.flatnonzero(in_box.values)
print(f"tag cells in receptor box: {len(box_tag_idx)}")

## Step 5 — Build a 30-year lazy graph and materialise once

Pangeo idiom: stack the 30 yearly stores into one virtual dataset
along a new `year` dim, sum over the box tags lazily (collapsing all
of central Chile into one moisture sink), and trigger a single
`.compute()` at the end with a progress bar. dask schedules the 30
reads concurrently.

We materialise three things at once:

- `year_src` — per-year 2-D source map (year, lat, lon)
- `year_land` — per-year land-only sum (year,)
- `year_total` — per-year total sum (year,)

In [ ]:
from dask.diagnostics import ProgressBar

ds_all = xr.concat(
    [
        xr.open_zarr(url, storage_options={"revision": HF_REVISION})
            .isel(tagging_mask=box_tag_idx)
        for url in AGUATRACK_YEARLY_URLS
    ],
    dim=xr.Variable("year", ALL_YEARS),
)
lsm = ds_ref.lsm

# Lazy: per-year 2-D source map.
year_src_lazy = ds_all.e_track.sum(dim="tagging_mask").mean(dim="time")  # (year, lat, lon)
# Lazy: scalar land + total annual sums.
year_total_lazy = year_src_lazy.sum(dim=("latitude", "longitude"))
year_land_lazy = year_src_lazy.where(lsm).sum(dim=("latitude", "longitude"))

with ProgressBar():
    year_src = year_src_lazy.load()
    year_total = year_total_lazy.load()
    year_land = year_land_lazy.load()

ds_ref.close()
year_ocean = year_total - year_land
year_ratio = year_land / year_total.where(year_total > 0)

print(f"loaded {year_src.sizes['year']} years "
      f"({year_src.nbytes / 1e6:.1f} MB in RAM)")

## Step 6 — Period source maps and anomaly

Average the source maps over the historical and drought periods, then
subtract to get the anomaly. `RdBu_r` (red = positive anomaly,
i.e. drought decade had *more* contribution from this pixel; blue =
negative anomaly, less) makes the spatial pattern of the deficit easy
to read.

In [ ]:
# Period composites via xarray slicing on the `year` coordinate.
hist_map = year_src.sel(year=HISTORICAL_YEARS).mean(dim="year")
drought_map = year_src.sel(year=DROUGHT_YEARS).mean(dim="year")
diff_map = drought_map - hist_map

# Shared colour-bar limits for the historical / drought panels.
vmax = float(max(hist_map.max(), drought_map.max()))
levels = np.linspace(0, vmax, 11)
# Symmetric limits for the anomaly panel — diverging colour map needs
# 0 at the centre.
diff_abs = float(max(abs(float(diff_map.min())), abs(float(diff_map.max()))))
diff_levels = np.linspace(-diff_abs, diff_abs, 11)


def style_ax(ax, title):
    """Decorate one panel of the 3-panel layout."""
    ax.coastlines(resolution="50m", color="black", linewidth=0.8)
    ax.set_extent([-90, -30, -55, 15], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.0)
    gl.top_labels = gl.right_labels = False
    # 20-degree spacing on lon, 10-degree on lat — keeps the panel
    # uncluttered.
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 20))
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, 10))
    ax.set_title(title)


fig, axes = plt.subplots(
    1, 3, figsize=(18, 5.3), subplot_kw={"projection": ccrs.PlateCarree()},
)
kw = dict(transform=ccrs.PlateCarree(), add_colorbar=False)
cf1 = hist_map.plot.contourf(ax=axes[0], levels=levels, cmap=cmc.batlowW_r, **kw)
drought_map.plot.contourf(ax=axes[1], levels=levels, cmap=cmc.batlowW_r, **kw)
cf3 = diff_map.plot.contourf(ax=axes[2], levels=diff_levels, cmap="RdBu_r", **kw)

# Outline the receptor box on every panel.
for ax in axes:
    ax.add_patch(mpatches.Rectangle(
        (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
        linewidth=2, edgecolor="red", facecolor="none",
        transform=ccrs.PlateCarree(), zorder=5,
    ))
style_ax(axes[0], "Historical Baseline (1990–2009)")
style_ax(axes[1], "Megadrought Period (2010–2019)")
style_ax(axes[2], "Anomaly (Drought - Historical)")

# Two horizontal colour bars: one shared between the first two panels,
# one for the anomaly panel only.
fig.subplots_adjust(bottom=0.14, wspace=0.05)
cbar1 = fig.add_axes([0.18, 0.05, 0.4, 0.02])
cbar2 = fig.add_axes([0.65, 0.05, 0.24, 0.02])
fig.colorbar(cf1, cax=cbar1, orientation="horizontal", label="Moisture Contribution (mm)")
fig.colorbar(cf3, cax=cbar2, orientation="horizontal", label="Anomaly (mm)")
fig.suptitle("Central Chile Moisture Sources — Megadrought Analysis", y=1.01)

OUT = Path("outputs/megadrought"); OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "fig1_source_maps.png", bbox_inches="tight", dpi=150)
plt.show()

## Step 7 — Ocean vs continental fraction by period

Same data as Step 5, aggregated to two periods and shown as fractions
of total moisture (so each period sums to 100% within its group).
Bars are paired: continental on the left of each pair, oceanic on the
right.

In [ ]:
def period_fraction(years):
    """Return (land_pct, ocean_pct) for a multi-year period.

    Pure xarray: slice the per-year DataArrays on `year`, take period
    means, and convert to percentages.
    """
    land = float(year_land.sel(year=years).mean())
    total = float(year_total.sel(year=years).mean())
    land_pct = land / total * 100
    return land_pct, 100.0 - land_pct


hist_land_pct, hist_ocean_pct = period_fraction(HISTORICAL_YEARS)
drought_land_pct, drought_ocean_pct = period_fraction(DROUGHT_YEARS)

fig, ax = plt.subplots(figsize=(11, 7))
x, w = np.arange(2), 0.35
ax.bar(x - w / 2, [hist_land_pct, drought_land_pct], w,
       label="Continental", color="#4a9e6b")
ax.bar(x + w / 2, [hist_ocean_pct, drought_ocean_pct], w,
       label="Oceanic (Pacific)", color="#3a7ebf")
ax.set_xticks(x)
ax.set_xticklabels(["Historical\n(1990–2009)", "Megadrought\n(2010–2019)"])
ax.set_ylabel("Moisture Fraction (%)")
ax.set_ylim(0, 100)
ax.set_title("Ocean vs Continental Moisture Sources\nCentral Chile (30-38 deg S)")
ax.legend()
fig.savefig(OUT / "fig2_ocean_land_bar.png", bbox_inches="tight", dpi=150)
plt.show()

## Step 8 — Annual stacked bar with recycling ratio overlay

Stacked bars (continental + oceanic) on the primary axis show the
absolute moisture contribution in mm/yr; the dark-red line on the
secondary axis tracks the continental-recycling ratio (%). Period
background shading and dashed period-mean lines highlight the regime
shift around 2010.

In [ ]:
# Pull the year axis once. Beyond this point we cross into matplotlib,
# which only speaks numpy — so .values is the right boundary call.
all_yrs = year_src.year.values
land_arr = year_land.values
ocean_arr = year_ocean.values
ratio_arr = (year_ratio * 100).values
y_max = float((year_land + year_ocean).max())

# Period-mean recycling ratios (shown as dashed lines below). xarray
# `.mean()` skips NaN by default.
hist_mean = float((year_ratio.sel(year=HISTORICAL_YEARS) * 100).mean())
drought_mean = float((year_ratio.sel(year=DROUGHT_YEARS) * 100).mean())

fig, ax1 = plt.subplots(figsize=(16, 6.9))

# Faint background shading over the two periods + a vertical line at
# the period boundary.
ax1.axvspan(1989.5, 2009.5, alpha=0.07, color="steelblue")
ax1.axvspan(2009.5, 2019.5, alpha=0.07, color="firebrick")
ax1.axvline(2009.5, color="black", lw=1.2, linestyle="--")

# Stacked bars: continental on the bottom, oceanic on top.
ax1.bar(all_yrs, land_arr, color="#4a9e6b", alpha=0.88, label="Continental (land)")
ax1.bar(all_yrs, ocean_arr, bottom=land_arr, color="#3a7ebf", alpha=0.88,
        label="Oceanic (Pacific)")

# Period text labels.
ax1.text(1999.5, y_max * 0.99, "Baseline\n1990–2009",
         ha="center", va="top", color="steelblue", fontweight="bold")
ax1.text(2014.5, y_max * 0.99, "Megadrought\n2010–2019",
         ha="center", va="top", color="firebrick", fontweight="bold")

ax1.set_xlabel("Year")
ax1.set_ylabel("Moisture Contribution (mm/yr)")
ax1.set_title("Annual Precipitation Origin — Central Chile (30-38 deg S)\n"
              "Stacked by Continental vs Oceanic Source")
ax1.legend(loc="upper left")
ax1.set_xlim(1989.5, 2019.5)

# Secondary axis for the recycling-ratio line.
ax2 = ax1.twinx()
ax2.plot(all_yrs, ratio_arr, color="darkred", lw=2, marker="o", ms=4,
         label="Recycling ratio")
ax2.set_ylabel("Continental Recycling Ratio (%)", color="darkred")
ax2.tick_params(axis="y", labelcolor="darkred")
# Generous headroom so the line and the period-mean dashed lines don't
# clip into the period text labels above.
ax2.set_ylim(0, max(float(np.nanmax(ratio_arr)) * 1.35, 10.0))

# Period-mean reference lines on the secondary axis. xmin/xmax are
# fractions of the axis width — we manually set them so each line spans
# only its own period.
n_hist = len(HISTORICAL_YEARS)
n_total = len(ALL_YEARS)
ax2.axhline(hist_mean, xmin=0.0, xmax=n_hist / n_total,
            color="darkred", lw=1.4, linestyle=":")
ax2.axhline(drought_mean, xmin=n_hist / n_total, xmax=1.0,
            color="darkred", lw=1.4, linestyle=":")
ax2.annotate(f"mean {hist_mean:.1f}%", xy=(HISTORICAL_YEARS[0], hist_mean),
             color="darkred", va="bottom", ha="left")
ax2.annotate(f"mean {drought_mean:.1f}%", xy=(DROUGHT_YEARS[0], drought_mean),
             color="darkred", va="bottom", ha="left")

fig.tight_layout()
fig.savefig(OUT / "fig3_annual_stacked_bar.png", bbox_inches="tight", dpi=150)
plt.show()

## Step 9 — Summary statistics

In [ ]:
total_hist = float(hist_map.sum())
total_drought = float(drought_map.sum())
deficit_pct = (total_drought - total_hist) / total_hist * 100

print("Period comparison (Central Chile, 30-38 deg S):\n")
print(f"  Total moisture input   : historical 100% -> drought {100 + deficit_pct:.1f}%  "
      f"({deficit_pct:+.1f}%)")
print(f"  Continental fraction   : {hist_land_pct:.1f}% -> {drought_land_pct:.1f}%  "
      f"({drought_land_pct - hist_land_pct:+.1f} pp)")
print(f"  Oceanic fraction       : {hist_ocean_pct:.1f}% -> {drought_ocean_pct:.1f}%  "
      f"({drought_ocean_pct - hist_ocean_pct:+.1f} pp)")
print(f"  Recycling-ratio mean   : {hist_mean:.1f}% -> {drought_mean:.1f}%  "
      f"({drought_mean - hist_mean:+.1f} pp)")